In [0]:
# Cell 1 — Install dependencies
%pip install langdetect "FlagEmbedding" --upgrade "transformers>=4.45" databricks-ai-search openai hf_transfer --quiet

In [ ]:
# Cell 2 — Imports + shared helpers
import os
import re
import sys
import time
import unicodedata
import warnings

import numpy as np
from FlagEmbedding import BGEM3FlagModel
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

# databricks.ai_search is notebook-scoped; extend the namespace path before import
import databricks
for _sp in sys.path:
    _dbp = os.path.join(_sp, "databricks")
    if os.path.isdir(os.path.join(_dbp, "ai_search")) and _dbp not in databricks.__path__:
        databricks.__path__.append(_dbp)
        break
from databricks.ai_search.client import AISearchClient

# langdetect is randomized by default; seed once here for reproducible output.
DetectorFactory.seed = 0


# ── Shared helpers (single source of truth for the AI Search cells) ─────────
def _already_exists(exc: Exception) -> bool:
    """True when an SDK create() failed only because the resource already exists.
    Message-based: the AI Search SDK exposes no typed 'AlreadyExists' error."""
    return "already exists" in str(exc).lower()


def _describe_state(desc: dict) -> str:
    """Normalized state string out of an AI Search index describe() dict."""
    st = (desc or {}).get("status", {}) or {}
    return str(
        st.get("detailed_state")
        or ("ONLINE_READY" if st.get("ready") is True else None)
        or st.get("message")
        or "UNKNOWN"
    )


In [ ]:
# Cell 3 — Config: catalog/schema, table names, lookup contract
# ---- Layer ----
CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"     
MART    = "gms_us_mart"

# ---- RAW reference inputs ----
RAW_CLINICAL = f"{CATALOG}.{ALYT}.Clinicalid_deviations"
RAW_DOCS = f"{CATALOG}.{ALYT}.documents_number_deviations"
RAW_ACRONYM  = f"{CATALOG}.{ALYT}.Acronyms_other_deviations"
RAW_CRO      = f"{CATALOG}.{ALYT}.cro_deviations"
RAW_DEVICE   = f"{CATALOG}.{ALYT}.rd_device_list_deviations"
RAW_VENDORS = f"{CATALOG}.{ALYT}.supplier_list_deviations"

# ---- SOURCE deviation data (read-only, in mart) ----
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"

# ---- OUTPUT lookups ----
REF_CLINICAL = f"{CATALOG}.{ALYT}.ref_clinical_norm"
REF_DOCS     = f"{CATALOG}.{ALYT}.ref_docs_norm"
REF_ACRONYM  = f"{CATALOG}.{ALYT}.ref_acronym_norm"
REF_CRO      = f"{CATALOG}.{ALYT}.ref_cro_norm"
REF_DEVICE   = f"{CATALOG}.{ALYT}.ref_device_norm"
REF_VENDORS = f"{CATALOG}.{ALYT}.ref_vendors_norm"
REF_UNIFIED  = f"{CATALOG}.{ALYT}.ref_glossary_unified"

# ---- The contract every lookup MUST expose (extra cols allowed) ----
LOOKUP_SCHEMA = ["key_norm", "entity_type", "canonical_id", "enrichment_text"]

# ---- Embedding + AI Search targets (canonical; restart-prone cells re-declare) ----
EMBED_INPUT    = f"{CATALOG}.{ALYT}.deviation_embed_input"
EMB_TABLE      = f"{CATALOG}.{ALYT}.deviation_embeddings"
EMB_INDEX_FINE = f"{CATALOG}.{ALYT}.deviation_emb_fine_idx"
EMB_INDEX_MID  = f"{CATALOG}.{ALYT}.deviation_emb_mid_idx"
EMB_INDEX_CORE = f"{CATALOG}.{ALYT}.deviation_emb_core_idx"
VS_ENDPOINT    = "deviation-retrieval-vs"
EMB_DIM        = 1024
MAX_LEN        = 8192          # BGE-M3 max sequence length

print("Config loaded.")
print("  Inputs :", RAW_CLINICAL, RAW_DOCS, RAW_ACRONYM, RAW_CRO, RAW_DEVICE, RAW_VENDORS, sep="\n           ")
print("  Source :", SOURCE_TABLE)
print("  Output :", REF_UNIFIED)


In [0]:
# Cell 4 — Shared UDFs (clinical / doc / acronym / CRO / device / vendor)
# ============================================================================
# SECTION A — CLINICAL IDs  (extract + parse; 3-source logic: free text + protocol + program)
# ============================================================================
CLINICAL_ID_ALTERNATIVES = [
    r"TAK[-\s_]?\d{2,4}[-_/]\d{3,4}",
    r"TAK[-\s_]?\d{2,4}",
    r"MLN[-\s_]?\d{3,4}[-_/]CCT-\d{2,4}",
    r"MLN[-\s_]?\d{3,4}[-_/]\d{2,4}",
    r"MLN[-\s_]?\d{3,4}",
    r"SHP[-\s_]?\d{3,4}[-_/]\d{2,4}",
    r"SHP[-\s_]?\d{3,4}",
    r"HGT[-\s_]?[A-Z]{2,4}[-_/]\d{2,4}",
    r"CCT[-_]?\d{2,4}",
    r"DEN[-_]?\d{2,4}",
    r"C\d{5}",
]
ID_REGEX = re.compile("|".join(CLINICAL_ID_ALTERNATIVES), flags=re.IGNORECASE)

# Program_Number filter: keep only values whose leading token is a valid clinical-ID shape.
PROGRAM_FILTER_STR = (
    r"^(TAK-?\d{2,4}|MLN\d{4}|SHP-?\d{3,4}|HGT-[A-Z]{2,4}-\d{2,4}"
    r"|C\d{5}|CCT-\d{2,4}|DEN-\d{2,4})"
)

def _which_prefix(u):
    for p in ("TAK", "MLN", "SHP", "HGT", "CCT", "DEN"):
        if u.startswith(p):
            return p
    if re.match(r"C\d{5}$", u):
        return "INTERNAL"
    return "UNKNOWN"

def parse_clinical_id(raw):
    u = re.sub(r"-{2,}", "-", re.sub(r"[\s_/]+", "-", str(raw).upper())).strip("-")
    prefix = _which_prefix(u)
    compound = suffix = None
    if prefix in ("TAK", "MLN", "SHP"):
        m = re.match(prefix + r"-?(\d+)(?:-(.+))?$", u)
        if m: compound, suffix = m.group(1), m.group(2)
        key = (prefix + "-" + compound) if compound else u
    elif prefix == "HGT":
        m = re.match(r"HGT-([A-Z]{2,4})-(\d+)$", u)
        if m: compound, suffix = m.group(1), m.group(2)
        key = ("HGT-" + compound) if compound else u
    elif prefix in ("CCT", "DEN"):
        m = re.match(prefix + r"-?(\d+)$", u)
        compound, suffix, key = prefix, (m.group(1) if m else None), u
    elif prefix == "INTERNAL":
        m = re.match(r"C(\d+)$", u)
        compound, key = (m.group(1) if m else None), u
    else:
        key = u
    return {"raw": raw, "canonical": u, "prefix": prefix,
            "compound_number": compound, "study_suffix": suffix, "normalized_key": key}

@F.udf(T.ArrayType(T.StringType()))
def extract_clinical_keys_udf(text):
    """(1) Free text: extract clinical IDs, return normalized compound+study keys."""
    if not text: return []
    keys = set()
    for m in ID_REGEX.finditer(str(text)):
        p = parse_clinical_id(m.group(0))
        if p["normalized_key"]: keys.add(p["normalized_key"])
        if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

@F.udf(T.ArrayType(T.StringType()))
def protocol_keys_udf(v):
    """(2) Study_Protocol: split on ';', TRUST ALL values -> normalized clinical keys."""
    if not v: return []
    _NA_PATTERNS = {"N/A", "NA", "N-A", "NONE", "NULL", "-", "--", ""}
    keys = set()
    for s in re.split(r"\s*;\s*", str(v)):
        s = s.strip()
        if not s or s.upper() in _NA_PATTERNS: continue
        p = parse_clinical_id(s)
        if p["normalized_key"]: keys.add(p["normalized_key"])
        if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

@F.udf(T.ArrayType(T.StringType()))
def program_keys_udf(v):
    """(3) Program_Number: split on ';', keep only valid clinical-ID patterns."""
    if not v: return []
    keys = set()
    for s in re.split(r"\s*;\s*", str(v)):
        s = s.strip()
        if s and re.match(PROGRAM_FILTER_STR, s):
            p = parse_clinical_id(s)
            if p["normalized_key"]: keys.add(p["normalized_key"])
            if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

def _strip_accents(s):
    """Fold accented Latin chars to ASCII (Genève -> Geneve). Non-Latin left as-is."""
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

def clean_freetext(v):
    """Case-PRESERVING cleanup: unicode hyphens/quotes/whitespace + accent folding.
       Do NOT lowercase — acronym & CRO extraction rely on original casing."""
    if not v:
        return None
    s = unicodedata.normalize("NFKC", str(v))
    s = re.sub(r"[\u2010-\u2015\u2212]", "-", s)   # unicode hyphens/minus -> '-'
    s = s.replace("\u00a0", " ")                    # non-breaking space -> space
    s = _strip_accents(s)                           # fold accents for matching
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip() or None

clean_freetext_udf = F.udf(clean_freetext, T.StringType())

# --- Language detection: keep only English (or code-only) events ---
def _detect_lang(text):
    """Return ISO code ('en', 'de', ...) or None. Short/ID-only ASCII text -> 'en'.
       Text containing CJK / substantial non-Latin script is NOT auto-whitelisted."""
    if not text or not text.strip():
        return None
    stripped = text.strip()

    # If there's any CJK / non-Latin letters, fall through to real detection.
    has_cjk = re.search(
        r"[\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff\uac00-\ud7af]", stripped
    )

    if not has_cjk:
        # ASCII-ish: allow the code-only escape hatch
        residual = re.sub(r"\b[A-Z]{2,}[-\s]?\d+\b", " ", stripped)
        if not re.search(r"[A-Za-z]{3,}", residual):
            return "en"

    try:
        DetectorFactory.seed = 0
        return detect(stripped)
    except LangDetectException:
        return None

def keep_if_english(text):
    """Return the text if English (or code-only), else None. Case-preserving."""
    lang = _detect_lang(text)
    return text if lang == "en" else None

keep_if_english_udf = F.udf(keep_if_english, T.StringType())

# ============================================================================
# SECTION B — DOCUMENTS  (regex EXACTLY matching SQL patterns)
# ============================================================================
DOC_PATTERNS = [
    r"\bSOP-\d+",
    r"\bSPEC-\d+",
    r"\bMTHD-\d+",
    r"\bMTD-\d+",      # legacy method prefix seen in your reference (MTD-002598)
    r"\bPROC-\d+",
    r"\bTOOL-\d+",
    r"\bFORM-\d+",
    r"\bWI-\d+",
]
DOC_REGEX = re.compile("|".join(DOC_PATTERNS), flags=re.IGNORECASE)
DOC_PREFIXES = r"(SOP|SPEC|MTHD|MTD|PROC|TOOL|FORM|WI)"

def norm_doc(v):
    """Canonical doc key. Note: MTD->MTHD collapse so legacy+current align."""
    if not v: return None
    u = re.sub(r"-{2,}", "-", re.sub(r"[\s_]+", "-", str(v).upper())).strip("-")
    if not re.match(DOC_PREFIXES + r"-?\d", u): return None
    u = re.sub(r"^MTD-", "MTHD-", u)          # unify legacy method prefix
    return u

norm_doc_udf = F.udf(norm_doc, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def extract_doc_keys_udf(text):
    """Extraction-side twin of your SQL: pull all doc IDs from text, normalized."""
    if not text: return []
    out = set()
    for m in DOC_REGEX.finditer(str(text)):
        k = norm_doc(m.group(0))
        if k: out.add(k)
    return list(out)

@F.udf(T.ArrayType(T.StringType()))
def legacy_doc_keys_udf(v):
    """Reference-side: parse messy legacy_numbers__c like
       'LSHIRE_1174436_6_0;;n/a;;TO SOP-0895' -> ['SOP-0895']."""
    if not v: return []
    out = set()
    for chunk in str(v).split(";;"):
        chunk = chunk.strip()
        if not chunk or chunk.lower() == "n/a": continue
        for m in re.finditer(DOC_PREFIXES + r"[-\s]?\d{3,7}", chunk, re.I):
            k = norm_doc(m.group(0))
            if k: out.add(k)
    return list(out)

# ============================================================================
# SECTION C — ACRONYMS  (regex candidate extraction + KNOWN-SET filter)
# ============================================================================
ACRONYM_CANDIDATE = re.compile(r"\(?[A-Z]{2,}\)?(?:[-/][A-Z0-9]+)?|\([A-Z]{2,}\)\s?[A-Z]{2,}")
ACR_DEF_JUNK = {"", "n/a", "na", "none", "null", "somebody", "unknown",
                "tbd", "todo", "test", "xxx", "-", "--", "."}

def norm_acr(v):
    if not v: return None
    u = re.sub(r"\s+", " ", str(v).strip()).upper()
    return u or None

norm_acr_udf = F.udf(norm_acr, T.StringType())

def make_extract_acronym_udf(known_keys):
    """Factory: returns a UDF that extracts only acronyms present in the known set.
       Min length 3 applied here to suppress short false-positive matches (e.g. 'IN', 'OF')."""
    known = frozenset(k for k in known_keys if k and len(k) >= 3)
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text: return []
        out = set()
        for m in ACRONYM_CANDIDATE.finditer(str(text)):
            k = norm_acr(m.group(0))
            if k and k in known:
                out.add(k)
        return list(out)
    return _udf

# ============================================================================
# SECTION D — DEVICES  (substring match on nonsensical names; context = type + owner)
# ============================================================================
# Business-owner abbreviation expansions (longest keys first at match time)
BUSINESS_OWNER_MAP = {
    "RGH TAU": "Rare Genetics and Hematology Therapeutic Area Unit",
    "GI TAU":  "Gastrointestinal Therapeutic Area Unit",
    "OTAU":    "Oncology Therapeutic Area Unit",
    "PDT":     "Plasma Derived Therapy",
}

def expand_business_owner(v):
    """Expand known business-owner abbreviations found anywhere in the value."""
    if v is None or str(v).strip() == "":
        return ""
    s = str(v)
    # replace longer keys first to avoid partial shadowing
    for abbr in sorted(BUSINESS_OWNER_MAP, key=len, reverse=True):
        s = re.sub(rf"\b{re.escape(abbr)}\b", BUSINESS_OWNER_MAP[abbr], s, flags=re.I)
    return s
expand_business_owner_udf = F.udf(expand_business_owner, T.StringType())

def norm_device(v):
    """Lowercase + collapse whitespace. Device names are matched as substrings,
       so we keep them permissive (no aggressive stripping)."""
    if not v:
        return None
    u = re.sub(r"\s+", " ", str(v)).strip().lower()
    return u or None

@F.udf(T.ArrayType(T.StringType()))
def device_variants_udf(name, alias, long_name):
    """Reference-side: normalized name/alias/long_name variants for the device pool."""
    return list({norm_device(x) for x in (name, alias, long_name) if x and norm_device(x)})

def make_extract_device_exact_udf(known_keys):
    """Substring matcher: return every device key that appears in the (lowercased) text.
       Mirrors the SQL `instr(lower(text), lower(name)) > 0` logic."""
    known = list(frozenset(k for k in known_keys if k and len(k) >= 4))  # skip 1-3 char noise
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text:
            return []
        low = str(text).lower()
        return [k for k in known if k in low]
    return _udf

print("Shared UDFs registered: clinical (3-source), doc, acronym(factory), cro, device, vendor.")

In [0]:
# Cell 5 — ReferenceBuilder class + build REF_ACRONYM

class ReferenceBuilder:
    """Shared builder for the normalized reference lookups.

    Entity-specific key/context extraction stays in each cell; this class owns
    the parts every builder repeats: wrapping into the
    `[TYPE] "key" / Canonical / <context>` enrichment contract, projecting to
    LOOKUP_SCHEMA, writing the table + comment, and reading back the key set.
    """

    def __init__(self, entity_type, table, comment):
        self.entity_type = entity_type
        self.table = table
        self.comment = comment

    def finalize(self, df, context=None):
        """Wrap key_norm/canonical_id/context into the shared enrichment contract.

        `df` must expose `key_norm` and `canonical_id`; `context` defaults to the
        df's existing `enrichment_text` column (the entity-specific context blob).
        """
        context = context if context is not None else F.col("enrichment_text")
        return (
            df.withColumn("entity_type", F.lit(self.entity_type))
              .withColumn("enrichment_text", F.concat(
                  F.lit(f'[{self.entity_type}] "'), F.col("key_norm"), F.lit('"\n'),
                  F.lit("Canonical: "), F.col("canonical_id"), F.lit("\n"),
                  context))
              .select(*LOOKUP_SCHEMA)
        )

    def write(self, df):
        """Overwrite the output table, attach the comment, and preview it."""
        (df.write.mode("overwrite").option("overwriteSchema", "true")
           .saveAsTable(self.table))
        spark.sql(f"COMMENT ON TABLE {self.table} IS '{self.comment}'")
        display(spark.table(self.table).limit(20))
        return spark.table(self.table)

    def key_set(self):
        """Distinct key_norm values for downstream substring/known-set extraction."""
        return set(r["key_norm"] for r in
                   spark.table(self.table).select("key_norm").collect())


ac = spark.table(RAW_ACRONYM)

# Guard: don't let short acronyms shadow real clinical/doc IDs
ID_LIKE = r"^(TAK|MLN|SHP|HGT|CCT|DEN|SOP|SPEC|MTHD|FORM|TOOL|WI|PROC)[-\s]?\d"

def _britfix(col):
    """Normalise British -ise/-isation spellings → American -ize/-ization.
       Applied before deduplication so near-identical spellings collapse to one."""
    for pat, rep in [('isation','ization'), (r'ised\b','ized'), (r'ising\b','izing'), (r'ise\b','ize')]:
        col = F.regexp_replace(col, pat, rep)
    return col

acronym_builder = ReferenceBuilder(
    "ACRONYM", REF_ACRONYM,
    "Sense-aware acronym lookup: key_norm→all definitions. Built from raw_acronym.")

ref_acronym = acronym_builder.finalize(
    ac.withColumn("key_norm", norm_acr_udf(F.col("Acronym")))
      .filter(F.col("key_norm").isNotNull())
      .filter(F.length("key_norm") >= 3)                      # drop 1-2 char noise
      .filter(~F.col("key_norm").rlike(ID_LIKE))              # don't shadow IDs
      .filter(F.col("Category").isin("Medical", "Industry")).filter(F.col("Definition").isNotNull())
      .filter(~F.lower(F.trim(F.col("Definition"))).isin(*ACR_DEF_JUNK))
      .filter(F.length(F.trim(F.col("Definition"))) >= 2)
      .groupBy("key_norm")
      .agg(
          F.collect_set(_britfix(F.lower(F.trim(F.col("Definition"))))).alias("definitions"),
          F.first("Acronym", ignorenulls=True).alias("canonical_id"),
      ),
    context=F.concat(
        F.lit("Definitions: {"),
        F.array_join(F.array_sort(F.col("definitions")), ", "),
        F.lit("}")),
)
acronym_builder.write(ref_acronym)

# Build a plain dict {ACRONYM_UPPER: first_definition} for TA/modality expansion in REF_CLINICAL
_acr_pd = (ac.filter(F.col("Category").isin("Medical", "Industry"))
             .select("Acronym", "Definition").toPandas())
ACR_MAP = {norm_acr(a): d for a, d in zip(_acr_pd["Acronym"], _acr_pd["Definition"]) if norm_acr(a)}

# Hand-curated TA overrides win over the generic acronym table
TA_OVERRIDES = {
    "NS": "Neuroscience", "ONC": "Oncology", "GI": "Gastroenterology",
    "RARE": "Rare Diseases", "PDT": "Plasma-Derived Therapies",
}
def expand_abbr(v):
    if v is None or str(v).strip() == "":
        return ""
    u = norm_acr(v)
    return TA_OVERRIDES.get(u) or ACR_MAP.get(u) or v   # fall back to original text
expand_abbr_udf = F.udf(expand_abbr, T.StringType())

print(f"Acronym map size: {len(ACR_MAP):,}")

In [0]:
# Cell 6 — Build acronym known-set extractor
acr_key_set = acronym_builder.key_set()
extract_acronym_keys_udf = make_extract_acronym_udf(acr_key_set)   # pass the plain set
print(f"Acronym known-set size: {len(acr_key_set):,}")

In [0]:
# Cell 7 — Build REF_CLINICAL
# ============================================================================
# REF_CLINICAL  (match on ALL alias cols; rich expanded context)
# ============================================================================
clin = spark.table(RAW_CLINICAL)
print("RAW_CLINICAL columns:", clin.columns)   # <- verify alias/context names

# --- Alias columns: any of these, if present in text, should resolve the row ---
CLIN_ALIAS_COLS = [
    "Parent_Node", "Name", "Protocol Number", "Alternate_Name",
    "Development_Name", "Parent_Alias", "Grand_Parent", "Grandparent_Alias",
]

def _clin_c(name):
    return F.col(f"`{name}`") if name in clin.columns else F.lit(None).cast("string")

# UDF: build normalized keys from every available alias value
@F.udf(T.ArrayType(T.StringType()))
def clinical_alias_keys_udf(*vals):
    keys = set()
    for v in vals:
        if v and str(v).strip():
            p = parse_clinical_id(str(v))
            if p["normalized_key"]: keys.add(p["normalized_key"])
            if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

# --- Context (expand TA/modality abbreviations) ---
clin_enriched = clin.withColumn(
    "enrichment_text",
    F.concat_ws(
        "\n",
        F.concat(F.lit("Generic Name: "),      F.coalesce(_clin_c("Generic_Name"), F.lit(""))),
        F.concat(F.lit("Modality: "),         expand_abbr_udf(_clin_c("Modality"))),
        F.concat(F.lit("Therapeutic Area: "), expand_abbr_udf(_clin_c("PF_TherapeuticArea"))),
        F.concat(F.lit("Finance TA Grouping: "), expand_abbr_udf(_clin_c("finance_TA_Grouping"))),
        F.concat(F.lit("Indication: "),       F.coalesce(_clin_c("IND_DESC"), F.lit(""))),
        F.concat(F.lit("Target: "),           F.coalesce(_clin_c("Target_Long_Name"), F.lit(""))),
        F.concat(F.lit("Mechanism: "),        F.coalesce(_clin_c("Mechanism"), F.lit(""))),
    ),
).withColumn(
    "canonical_id",
    F.coalesce(_clin_c("Development_Name"), _clin_c("Name"), _clin_c("Protocol Number")),
)

clinical_builder = ReferenceBuilder(
    "CLINICAL_ID", REF_CLINICAL,
    "Clinical-ID lookup keyed on all alias columns; context expanded via acronym map.")

alias_cols_present = [c for c in CLIN_ALIAS_COLS if c in clin.columns]
ref_clinical = clinical_builder.finalize(
    clin_enriched
    .withColumn("key_norm", F.explode(
        clinical_alias_keys_udf(*[F.col(f"`{c}`") for c in alias_cols_present])))
    .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
).dropDuplicates(["key_norm"])
clinical_builder.write(ref_clinical)

In [0]:
# Cell 8 — Build REF_DOCS
# ============================================================================
# REF_DOCS  (keys from doc# + previous# + legacy; context = title)
# ============================================================================
docs = spark.table(RAW_DOCS)
print("RAW_DOCS columns:", docs.columns)   # <- verify names below

DOC_NUM_COL    = "document_number__v"
DOC_PREV_COL   = "previous_document_number__c"
DOC_LEGACY_COL = "legacy_numbers__c"
DOC_TITLE_COL  = "title__v"

def _dc(name):
    return F.col(f"`{name}`") if name in docs.columns else F.lit(None).cast("string")

docs_enriched = docs.withColumn(
    "enrichment_text",
    F.concat(F.lit("Document Title: "), F.coalesce(_dc(DOC_TITLE_COL), F.lit(""))),
).withColumn("canonical_id", _dc(DOC_NUM_COL))

# straightforward single-value doc columns -> norm_doc
def _doc_keys_from(col_name):
    if col_name not in docs.columns:
        return None
    return (docs_enriched
            .withColumn("key_norm", norm_doc_udf(F.col(f"`{col_name}`")))
            .filter(F.col("key_norm").isNotNull()))

parts = [df for df in (_doc_keys_from(DOC_NUM_COL), _doc_keys_from(DOC_PREV_COL)) if df is not None]

# messy legacy column -> legacy_doc_keys_udf (explode)
if DOC_LEGACY_COL in docs.columns:
    parts.append(
        docs_enriched
        .withColumn("key_norm", F.explode(legacy_doc_keys_udf(F.col(f"`{DOC_LEGACY_COL}`"))))
        .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
    )

ref_docs_union = parts[0]
for df in parts[1:]:
    ref_docs_union = ref_docs_union.unionByName(df)

docs_builder = ReferenceBuilder(
    "DOCUMENT", REF_DOCS,
    "Doc lookup keyed on document_number__v + previous + legacy; context = title__v.")

# Keep the longest enrichment_text per key_norm (most informative context)
ref_docs = (
    docs_builder.finalize(ref_docs_union)
    .withColumn("_len", F.length("enrichment_text"))
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("key_norm").orderBy(F.col("_len").desc())))
    .filter(F.col("_rn") == 1)
    .select(*LOOKUP_SCHEMA)
)
docs_builder.write(ref_docs)

In [0]:
# Cell 9 — Build REF_CRO
# ============================================================================
# REF_CRO  (direct name lookup; context = name + display name + description)
# ============================================================================
cro = spark.table(RAW_CRO)
print("RAW_CRO columns:", cro.columns)

CRO_NM_COL   = "Name"
CRO_DISP_COL = "Display Name"
CRO_DESC_COL = "description"

def _cc(name):
    return F.col(f"`{name}`") if name in cro.columns else F.lit(None).cast("string")

cro_enriched = (
    cro
    .filter(_cc(CRO_NM_COL).isNotNull() & (F.trim(_cc(CRO_NM_COL)) != ""))
    .filter(~_cc(CRO_NM_COL).startswith("***"))
    .filter(_cc(CRO_NM_COL).rlike('^[\\x20-\\x7E]+$'))  # drop non-ASCII / non-English names
    .withColumn(
        "enrichment_text",
        F.concat_ws(
            "\n",
            F.concat(F.lit("CRO Name: "), F.coalesce(_cc(CRO_NM_COL), F.lit(""))),
            F.concat(F.lit("Display Name: "), F.coalesce(_cc(CRO_DISP_COL), F.lit(""))),
            F.concat(F.lit("Description: "), F.coalesce(_cc(CRO_DESC_COL), F.lit(""))),
        ),
    )
    .withColumn("canonical_id", _cc(CRO_NM_COL))
)

cro_builder = ReferenceBuilder(
    "CRO", REF_CRO,
    "CRO lookup: name used directly as key; context = CRO name + display name + description.")

ref_cro = cro_builder.finalize(
    cro_enriched
    .withColumn("key_norm", F.lower(F.trim(_cc(CRO_NM_COL))))
    .filter(F.col("key_norm").isNotNull() & (F.length("key_norm") >= 4))
    .groupBy("key_norm")
    .agg(
        F.first("canonical_id", ignorenulls=True).alias("canonical_id"),
        F.first("enrichment_text", ignorenulls=True).alias("enrichment_text"),
    )
)
cro_builder.write(ref_cro)

print(f"CRO pool size: {spark.table(REF_CRO).count():,}")

# Build CRO key set for exact substring extraction (same pattern as device/vendor)
cro_key_set = cro_builder.key_set()
extract_cro_exact_udf = make_extract_device_exact_udf(cro_key_set)
print(f"CRO known-set size: {len(cro_key_set):,}")

In [0]:
# Cell 10 — Build REF_DEVICE
# ============================================================================
# REF_DEVICE  (substring name pool; context = Device_Type + business_owner)
# ============================================================================
dev = spark.table(RAW_DEVICE)
print("RAW_DEVICE columns:", dev.columns)   # <- verify names below

DEV_NAME_COL   = "Name"
DEV_ALIAS_COL  = "Alias"
DEV_LONG_COL   = "Device_Long_name"
DEV_TYPE_COL   = "Device_Type"
DEV_OWNER_COL  = "business_owner"

def _dvc(name):
    return F.col(f"`{name}`") if name in dev.columns else F.lit(None).cast("string")

dev_enriched = (
    dev
    .withColumn(
        "enrichment_text",
            F.concat_ws(
                "\n",
                F.concat(F.lit("Device Name: "),  F.coalesce(_dvc(DEV_NAME_COL), F.lit(""))),
                F.concat(F.lit("Device Type: "),  F.coalesce(_dvc(DEV_TYPE_COL), F.lit(""))),
                F.concat(F.lit("Business Owner: "), expand_business_owner_udf(_dvc(DEV_OWNER_COL))),
            ),
    )
    .withColumn("canonical_id",
                F.coalesce(_dvc(DEV_LONG_COL), _dvc(DEV_NAME_COL)))
)

device_builder = ReferenceBuilder(
    "DEVICE", REF_DEVICE,
    "Device lookup: normalized name/alias/long_name variants matched as substrings; "
    "context = Device_Type + business_owner (abbreviations expanded).")

ref_device = device_builder.finalize(
    dev_enriched
    .withColumn("key_norm", F.explode(
        device_variants_udf(_dvc(DEV_NAME_COL), _dvc(DEV_ALIAS_COL), _dvc(DEV_LONG_COL))))
    .filter(F.col("key_norm").isNotNull() & (F.length("key_norm") >= 4))
).dropDuplicates(["key_norm"])
device_builder.write(ref_device)

# read back for the substring extractor
device_key_set = device_builder.key_set()
extract_device_exact_udf = make_extract_device_exact_udf(device_key_set)
print(f"Device known-set size: {len(device_key_set):,}")

In [0]:
# Cell 11 — Build REF_VENDORS
# ============================================================================
# REF_VENDORS  (direct name lookup; context = vendor name + country)
# ============================================================================
vnd = spark.table(RAW_VENDORS)
print("RAW_VENDORS columns:", vnd.columns)

VND_NM_COL      = "Supplier_Name_RD"
VND_COUNTRY_COL = "Country_RD"

def _vc(name):
    return F.col(f"`{name}`") if name in vnd.columns else F.lit(None).cast("string")

vnd_enriched = (
    vnd
    .filter(_vc(VND_NM_COL).isNotNull() & (F.trim(_vc(VND_NM_COL)) != ""))
    .withColumn(
        "enrichment_text",
        F.concat_ws(
            "\n",
            F.concat(F.lit("Vendor (External Supplier): "), F.coalesce(_vc(VND_NM_COL), F.lit(""))),
            F.concat(F.lit("Country: "), F.coalesce(_vc(VND_COUNTRY_COL), F.lit(""))),
        ),
    )
    .withColumn("canonical_id", _vc(VND_NM_COL))
)

vendor_builder = ReferenceBuilder(
    "VENDOR", REF_VENDORS,
    "Vendor lookup: supplier name used directly as key; context = vendor name + country.")

ref_vendors = vendor_builder.finalize(
    vnd_enriched
    .withColumn("key_norm", F.lower(F.trim(_vc(VND_NM_COL))))
    .filter(F.col("key_norm").isNotNull() & (F.length("key_norm") >= 4))
).dropDuplicates(["key_norm"])
vendor_builder.write(ref_vendors)

print(f"Vendor pool size: {spark.table(REF_VENDORS).count():,}")

# Build vendor key set for exact substring extraction (same pattern as device)
vendor_key_set = vendor_builder.key_set()
extract_vendor_exact_udf = make_extract_device_exact_udf(vendor_key_set)
print(f"Vendor known-set size: {len(vendor_key_set):,}")

In [0]:
# Cell 12 — Build REF_UNIFIED
# ============================================================================
# REF_UNIFIED  (union of all lookups; single table to join against)
# ============================================================================
ref_unified = (
    spark.table(REF_CLINICAL).select(*LOOKUP_SCHEMA)
    .unionByName(spark.table(REF_DOCS).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_ACRONYM).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_CRO).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_DEVICE).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_VENDORS).select(*LOOKUP_SCHEMA))
    .dropDuplicates(["key_norm", "entity_type"])
)


(ref_unified.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_UNIFIED))
spark.sql(f"COMMENT ON TABLE {REF_UNIFIED} IS "
          "'Unified glossary: clinical + document + acronym + CRO + device + vendor lookups, one row per (key_norm, entity_type).'")

print("Row counts per entity_type:")
display(spark.table(REF_UNIFIED).groupBy("entity_type").count())
display(spark.table(REF_UNIFIED).limit(20))

In [0]:
# Cell 13 — Source resolution + build embed input
# ============================================================================
# SOURCE RESOLUTION  (COMPRESS TO EVENT GRAIN FIRST)
# Builds deviation_embed_input with the contextual-retrieval schema:
#   source_free_text_{full,mid,core}  — cleaned deviation text (3 tiers)
#   deterministic_context             — resolved reference entities (was injected_context)
# LLM context + contextual_retrieval_text are added in the next cell.
# ============================================================================
src = spark.table(SOURCE_TABLE)
print("SOURCE columns:", src.columns)

SRC_ID_COL       = "Event_Number"
SRC_PROTOCOL_COL = "Study_Protocol"
SRC_PROGRAM_COL  = "Program_Number"
SRC_DOC_COL      = "Document_or_Process"

SRC_CORE_COLS   = ["Event_Title", "Event_Description"]
SRC_EVENT_COLS  = ["Impact_Assessment", "Quality_Final_Assessment",
                   "Root_Cause_Category", "Root_Cause_SubCategory"]
SRC_ROWGRAIN_COLS = ["Action_Text"]
SRC_MID_COLS = SRC_CORE_COLS + SRC_EVENT_COLS   # mid tier = core + event (no action)


def _sc(name):
    return F.col(f"`{name}`") if name in src.columns else F.lit(None).cast("string")

# ---- 1. COMPRESS to one row per Event_Number ----
agg_exprs = []
# event-stable columns -> first non-null
for c in SRC_CORE_COLS + SRC_EVENT_COLS + [SRC_PROTOCOL_COL, SRC_PROGRAM_COL, SRC_DOC_COL]:
    if c in src.columns:
        agg_exprs.append(F.first(_sc(c), ignorenulls=True).alias(c))
# row-grain columns -> distinct NON-NULL values joined into one string
for c in SRC_ROWGRAIN_COLS:
    if c in src.columns:
        # collect_list already drops nulls; array_join avoids empty-string artifacts
        agg_exprs.append(
            F.array_join(F.array_distinct(F.collect_list(_sc(c))), "\n").alias(c)
        )

src_event = (
    src.groupBy(F.col(f"`{SRC_ID_COL}`").cast("string").alias("pr_id"))
       .agg(*agg_exprs)
)
print(f"Compressed source: {src.count():,} rows -> {src_event.count():,} events")

# ---- 2. Clean text columns + language filter (keep English) + CJK guard ----
def _ec(name):
    return F.col(f"`{name}`") if name in src_event.columns else F.lit("").cast("string")

FULL_TEXT_COLS = SRC_CORE_COLS + SRC_EVENT_COLS + SRC_ROWGRAIN_COLS  # combined_text scope

# CJK regex — second-pass guard for anything the language filter lets through
_CJK_PATTERN = r"[\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff\uac00-\ud7af]"

src_base = (
    src_event.select(
        "pr_id",
        _ec(SRC_PROTOCOL_COL).alias("study_protocol"),
        _ec(SRC_PROGRAM_COL).alias("program_number"),
        _ec(SRC_DOC_COL).alias("document_or_process"),
        *[clean_freetext_udf(_ec(c)).alias(f"__cl_{c}") for c in FULL_TEXT_COLS],
    )
    # 1. Drop non-English events (langdetect on the core title + description)
    .filter(
        keep_if_english_udf(
            F.concat_ws(" ",
                        F.coalesce(F.col("__cl_Event_Title"), F.lit("")),
                        F.coalesce(F.col("__cl_Event_Description"), F.lit("")))
        ).isNotNull()
    )
    # 2. Belt-and-suspenders: drop any remaining CJK in title / description
    .filter(
        ~F.coalesce(F.col("__cl_Event_Title"), F.lit("")).rlike(_CJK_PATTERN) &
        ~F.coalesce(F.col("__cl_Event_Description"), F.lit("")).rlike(_CJK_PATTERN)
    )
    .withColumn(
        "combined_text",   # full context: core + event + action_text
        F.concat_ws("\n",
            *[F.concat(F.lit(f"{c}: "), F.coalesce(F.col(f"__cl_{c}"), F.lit("")))
              for c in FULL_TEXT_COLS]),
    )
    .withColumn(
        "core_text",       # title + description only
        F.concat_ws("\n",
            *[F.concat(F.lit(f"{c}: "), F.coalesce(F.col(f"__cl_{c}"), F.lit("")))
              for c in SRC_CORE_COLS]),
    )
    .withColumn(
        "mid_text",        # core + impact + quality + root-cause (no action)
        F.concat_ws("\n",
            *[F.concat(F.lit(f"{c}: "), F.coalesce(F.col(f"__cl_{c}"), F.lit("")))
              for c in SRC_MID_COLS]),
    )
    .select("pr_id", "combined_text", "mid_text", "core_text",
            "study_protocol", "program_number", "document_or_process")
    .filter(F.length(F.trim(F.col("combined_text"))) > 0)
)

# --- Extract all entity keys (free text + structured columns) ---
src_keys = (
    src_base
    .withColumn("clinical_keys",   extract_clinical_keys_udf(F.col("combined_text")))
    .withColumn("protocol_keys",   protocol_keys_udf(F.col("study_protocol")))
    .withColumn("program_keys",    program_keys_udf(F.col("program_number")))
    .withColumn("doc_keys",        extract_doc_keys_udf(F.col("combined_text")))          # free text
    .withColumn("doc_struct_keys", extract_doc_keys_udf(F.col("document_or_process")))    # structured
    .withColumn("acr_keys",        extract_acronym_keys_udf(F.col("combined_text")))
    .withColumn("device_keys",     extract_device_exact_udf(F.col("combined_text")))
    .withColumn("vendor_keys",     extract_vendor_exact_udf(F.col("combined_text")))
    .withColumn("cro_keys",        extract_cro_exact_udf(F.col("combined_text")))
)

# ---- Vendor + CRO extraction via exact substring match (same pattern as device) ----
# No n-grams needed; extract_vendor_exact_udf / extract_cro_exact_udf scan text for known names

def _explode_keys(df, arr_col, etype):
    return (df.select("pr_id", F.explode(F.col(arr_col)).alias("key_norm"))
              .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
              .withColumn("entity_type", F.lit(etype)))

exploded = (
    _explode_keys(src_keys, "clinical_keys",   "CLINICAL_ID")
    .unionByName(_explode_keys(src_keys, "protocol_keys",   "CLINICAL_ID"))
    .unionByName(_explode_keys(src_keys, "program_keys",    "CLINICAL_ID"))
    .unionByName(_explode_keys(src_keys, "doc_keys",        "DOCUMENT"))    # free text
    .unionByName(_explode_keys(src_keys, "doc_struct_keys", "DOCUMENT"))    # structured
    .unionByName(_explode_keys(src_keys, "acr_keys",        "ACRONYM"))
    .unionByName(_explode_keys(src_keys, "cro_keys",        "CRO"))
    .unionByName(_explode_keys(src_keys, "device_keys",     "DEVICE"))
    .unionByName(_explode_keys(src_keys, "vendor_keys",     "VENDOR"))
    .dropDuplicates(["pr_id", "key_norm", "entity_type"])
)

ref = spark.table(REF_UNIFIED).select(
    "key_norm", "entity_type",
    F.col("enrichment_text").alias("ref_enrichment"),
)

resolved = exploded.join(ref, on=["key_norm", "entity_type"], how="left")

# ---- deterministic_context: resolved reference entities, one blob per pr_id ----
#      (this is the STEP-1 "deterministic expanded entities" from the article diagram)
deterministic_per_pr = (
    resolved.filter(F.col("ref_enrichment").isNotNull())
    .groupBy("pr_id")
    .agg(F.concat_ws("\n---\n", F.collect_set("ref_enrichment")).alias("deterministic_context"))
)

# ---- Assemble source_free_text at three tiers (NO fusion yet) ----
#      full = title+desc+event+action | mid = title+desc+event | core = title+desc
embed_ready = (
    src_base.join(deterministic_per_pr, on="pr_id", how="left")
    .withColumn("deterministic_context",
                F.coalesce(F.col("deterministic_context"), F.lit("")))
    .withColumnRenamed("combined_text", "source_free_text_full")
    .withColumnRenamed("mid_text",      "source_free_text_mid")
    .withColumnRenamed("core_text",     "source_free_text_core")
    .select(
        "pr_id",
        "source_free_text_full", "source_free_text_mid", "source_free_text_core",
        "deterministic_context",
    )
)

# ---- Write via staging (safe on reruns where embed_input already exists) ----
EMBED_INPUT = f"{CATALOG}.{ALYT}.deviation_embed_input"
STAGING     = f"{CATALOG}.{ALYT}.deviation_embed_input_staging"

(embed_ready.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(STAGING))
(spark.table(STAGING).write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(EMBED_INPUT))
spark.sql(f"DROP TABLE IF EXISTS {STAGING}")

print("Coverage — deviations with >=1 resolved reference:")
display(embed_ready.select(
    F.count("*").alias("total"),
    F.sum(F.when(F.length("deterministic_context") > 0, 1).otherwise(0)).alias("with_context"),
))
display(spark.table(EMBED_INPUT).limit(10))

In [0]:
# Cell 14 — Diagnostic: extraction + join coverage
# DIAGNOSTIC — how many source rows extract each entity type, and how many join
print("Source rows with ≥1 extracted key (pre-join, by entity type):")
display(
    exploded.groupBy("entity_type")
    .agg(F.countDistinct("pr_id").alias("rows_with_key"))
    .orderBy("entity_type")
)

print("Exploded keys that actually matched REF_UNIFIED:")
display(
    exploded.join(ref, on=["key_norm", "entity_type"], how="left")
    .groupBy("entity_type")
    .agg(
        F.count("*").alias("extracted"),
        F.sum(F.when(F.col("ref_enrichment").isNotNull(), 1).otherwise(0)).alias("matched"),
    )
)

print("Top extracted keys that FAILED to match (fix these first):")
display(
    exploded.join(ref, on=["key_norm", "entity_type"], how="left")
    .filter(F.col("ref_enrichment").isNull())
    .groupBy("entity_type", "key_norm").count()
    .orderBy(F.col("count").desc()).limit(30)
)

In [0]:
# Cell 15 — Generate LLM situating context (ai_query)
# ============================================================================
# LLM CONTEXT  (Contextual Retrieval "situating context")
#
# STEP 2 of the article pipeline. For each deviation, ask a Databricks-hosted
# foundation model to write a short paragraph that situates the raw free text
# within its resolved deterministic entities (clinical IDs, docs, acronyms,
# CROs, devices, vendors).
#
# Runs SERVER-SIDE via ai_query() — no local vLLM, no torch/CUDA/GPU state,
# no restartPython() risk. Auto-parallelized across the endpoint.
#
# NOTE: this DBR's ai_query 'request' arg accepts StringType ONLY, so the
# system + user prompts are concatenated into a single prompt string
# (the messages-array form raised AI_FUNCTION_UNSUPPORTED_REQUEST).
#
# Adds to deviation_embed_input:
#   llm_context
#   contextual_retrieval_text_{full,mid,core}
#     = deterministic_context + llm_context + source_free_text_<tier>
# ============================================================================

EMBED_INPUT = f"{CATALOG}.{ALYT}.deviation_embed_input"          # ← final target (unchanged)
STAGING     = f"{CATALOG}.{ALYT}.deviation_embed_input_stg"      # ← temp, dropped at end
FM_MODEL    = "databricks-gpt-5-4-mini"

# ── 1. Build ONE prompt string (system + user) — ai_query wants StringType ─
prompt_col = F.concat(
    # system portion
    F.lit("You situate clinical-trial deviation records for search retrieval.\n"
          "<deterministic_context>\n"),
    F.coalesce("deterministic_context", F.lit("")),
    F.lit("\n</deterministic_context>\n\n"),
    # user portion
    F.lit("Here is the deviation free text we want to situate.\n<deviation_text>\n"),
    F.coalesce("source_free_text_full", F.lit("")),
    F.lit("\n</deviation_text>\n\n"
          "Using the deterministic_context provided (resolved clinical IDs, documents,\n"
          "acronyms, CROs, devices, and vendors), write a short succinct paragraph that\n"
          "situates this deviation: which entities it concerns and what it is about, so\n"
          "that it can be retrieved more accurately. Answer only with the succinct\n"
          "context and nothing else."),
)

# ── 2. Generate llm_context via ai_query (string request; skip empty rows) ─
enriched = (spark.table(EMBED_INPUT)
    .withColumn("_prompt", prompt_col)
    .withColumn("llm_context",
        F.when(F.length(F.trim(F.coalesce("source_free_text_full", F.lit("")))) == 0,
               F.lit(""))
         .otherwise(F.expr(f"""
            ai_query(
              '{FM_MODEL}',
              request => _prompt,
              modelParameters => named_struct('max_tokens', 300, 'temperature', 0.0)
            )
         """)))
    .drop("_prompt"))

# ── 3. Fuse tiers: deterministic + llm + source (concat_ws drops nulls) ────
def _fuse(tier_col):
    return F.concat_ws("\n\n",
        F.when(F.length(F.trim(F.coalesce("deterministic_context", F.lit("")))) > 0,
               F.concat(F.lit("=== RESOLVED REFERENCES ===\n"), F.col("deterministic_context"))),
        F.when(F.length(F.trim(F.coalesce("llm_context", F.lit("")))) > 0,
               F.concat(F.lit("=== CONTEXT ===\n"), F.col("llm_context"))),
        F.col(tier_col))

enriched = (enriched
    .withColumn("contextual_retrieval_text_full", _fuse("source_free_text_full"))
    .withColumn("contextual_retrieval_text_mid",  _fuse("source_free_text_mid"))
    .withColumn("contextual_retrieval_text_core", _fuse("source_free_text_core")))

# ── 4. Enforce the exact same column order/schema as the original cell ─────
final_cols = [
    "pr_id",
    "source_free_text_full", "source_free_text_mid", "source_free_text_core",
    "deterministic_context", "llm_context",
    "contextual_retrieval_text_full", "contextual_retrieval_text_mid", "contextual_retrieval_text_core",
]
enriched = enriched.select(*final_cols)

# ── 5. Write: materialize to STAGING first so ai_query runs ONCE, then swap ─
(enriched.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(STAGING))
(spark.table(STAGING).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(EMBED_INPUT))
spark.sql(f"DROP TABLE IF EXISTS {STAGING}")

print(f"{EMBED_INPUT} now has: source_free_text_{{full,mid,core}} | "
      f"deterministic_context | llm_context | contextual_retrieval_text_{{full,mid,core}}")
display(spark.table(EMBED_INPUT).limit(5))

In [0]:
# Cell 16 — Verify llm_context populated
# ============================================================================
# VERIFY llm_context populated + print readable samples
# ============================================================================

EMBED_INPUT = f"{CATALOG}.{ALYT}.deviation_embed_input"
t = spark.table(EMBED_INPUT)

total     = t.count()
non_empty = t.filter(F.length(F.trim(F.coalesce("llm_context", F.lit("")))) > 0).count()
empty     = total - non_empty

print(f"Rows total ...................... {total:,}")
print(f"llm_context NON-EMPTY ........... {non_empty:,}  ({100*non_empty/total:.1f}%)")
print(f"llm_context empty (skipped) ..... {empty:,}\n")

# ── Pull a few populated rows to the driver and print them plainly ─────────
samples = (t.filter(F.length(F.trim(F.coalesce("llm_context", F.lit("")))) > 0)
             .select("pr_id", "llm_context",
                     F.length("contextual_retrieval_text_full").alias("ctx_full_chars"))
             .limit(3)
             .collect())

for i, r in enumerate(samples, 1):
    print("="*90)
    print(f"SAMPLE {i}   pr_id={r['pr_id']}   ctx_full_chars={r['ctx_full_chars']:,}")
    print("-"*90)
    print(r["llm_context"])
    print()

In [0]:
# Cell 17 — Build BGE-M3 embeddings table (three tiers)
# ============================================================================
# SECTION I — BGE-M3 EMBEDDINGS TABLE  (three tiers)  [DEVIATIONS]
#
# STEP 3 of the article pipeline. Embeds contextual_retrieval_text
# (= deterministic_context + llm_context + source_free_text) at three tiers:
#
#   embedding      (fine) — contextual_retrieval_text_full
#   mid_embedding  (mid)  — contextual_retrieval_text_mid
#   core_embedding (core) — contextual_retrieval_text_core
#
# Keyed on pr_id (Event_Number). One BGE-M3 pass over [full|mid|core] -> split.
#
# MODEL: BAAI/bge-m3  |  backbone XLM-RoBERTa-large  |  max supported len = 8192
# ============================================================================

EMBED_INPUT     = f"{CATALOG}.{ALYT}.deviation_embed_input"   # <- Cell 13/15 output
EMB_TABLE       = f"{CATALOG}.{ALYT}.deviation_embeddings"
EMB_DIM         = 1024
MAX_LEN         = 8192         # BGE-M3 supports up to 8192
BATCH           = 64           # long sequences need smaller batches

# The three contextual_retrieval_text tiers we embed (fine / mid / core).
TIER_COLS = [
    "contextual_retrieval_text_full",
    "contextual_retrieval_text_mid",
    "contextual_retrieval_text_core",
]

# ── 1. Load BGE-M3 on GPU driver (FP16 for H100) ──────────────────────────
print("Loading BAAI/bge-m3 (first run downloads ~3 GB from HuggingFace) …")
bge = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("Model loaded.")

# ── 2. Pull ENRICHED DEVIATIONS to driver ─────────────────────────────────
src_pd = (
    spark.table(EMBED_INPUT)
    .select(
        F.col("pr_id"),
        "source_free_text_full", "source_free_text_mid", "source_free_text_core",
        "deterministic_context", "llm_context",
        *TIER_COLS,
    )
    .toPandas()
)
src_pd["pr_id"] = src_pd["pr_id"].astype(str)

# Safety: never feed None to the encoder
for c in (["source_free_text_full", "source_free_text_mid", "source_free_text_core",
           "deterministic_context", "llm_context"] + TIER_COLS):
    src_pd[c] = src_pd[c].fillna("")

n = len(src_pd)
print(f"Rows: {n:,}  |  Total encode calls: {n * 3:,} (3 tiers in one pass)")

# ── 3. Single encode pass — all three text lists concatenated ─────────────
# Concatenate [fine | mid | core], encode once, then split by n.
all_texts = (
    src_pd["contextual_retrieval_text_full"].tolist()
    + src_pd["contextual_retrieval_text_mid"].tolist()
    + src_pd["contextual_retrieval_text_core"].tolist()
)

# ---- Length-sort batching -------------------------------------------------
# Padding is per-batch: one long row pads its whole batch up. Sort by token
# length so long rows batch together and short rows stay short.
tok = bge.tokenizer
approx_len = [len(tok.encode(t, add_special_tokens=True)) for t in all_texts]
order      = sorted(range(len(all_texts)), key=lambda i: approx_len[i])
inv        = np.argsort(order)                       # to restore original order
all_texts_sorted = [all_texts[i] for i in order]

all_vecs_sorted = []
t0 = time.time()
for i in range(0, len(all_texts_sorted), BATCH):
    chunk = all_texts_sorted[i : i + BATCH]
    out = bge.encode(
        chunk,
        batch_size=len(chunk),
        max_length=MAX_LEN,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    all_vecs_sorted.extend(out["dense_vecs"].tolist())

# Restore original ordering
all_vecs = [all_vecs_sorted[inv[i]] for i in range(len(all_texts))]

print(f"Encoded {len(all_vecs):,} vectors in {time.time()-t0:.1f}s  |  dim={len(all_vecs[0])}")
assert len(all_vecs[0]) == EMB_DIM

# Split back into three tiers
src_pd["embedding"]      = all_vecs[:n]
src_pd["mid_embedding"]  = all_vecs[n : 2 * n]
src_pd["core_embedding"] = all_vecs[2 * n :]

# ── 4. Write Delta table with CDF + PRIMARY KEY (pr_id) ─────────────────────
emb_schema = T.StructType([
    T.StructField("pr_id",                          T.StringType(),             False),  # <- PK
    T.StructField("source_free_text_full",          T.StringType(),             True),
    T.StructField("source_free_text_mid",           T.StringType(),             True),
    T.StructField("source_free_text_core",          T.StringType(),             True),
    T.StructField("deterministic_context",          T.StringType(),             True),
    T.StructField("llm_context",                    T.StringType(),             True),
    T.StructField("contextual_retrieval_text_full", T.StringType(),             True),
    T.StructField("contextual_retrieval_text_mid",  T.StringType(),             True),
    T.StructField("contextual_retrieval_text_core", T.StringType(),             True),
    T.StructField("embedding",                      T.ArrayType(T.FloatType()), True),
    T.StructField("mid_embedding",                  T.ArrayType(T.FloatType()), True),
    T.StructField("core_embedding",                 T.ArrayType(T.FloatType()), True),
])
cols = [f.name for f in emb_schema.fields]

# Write in chunks to stay under Spark Connect's 3GB local relation limit
CHUNK_SIZE = 50_000
for i in range(0, len(src_pd), CHUNK_SIZE):
    chunk_pdf = src_pd[cols].iloc[i : i + CHUNK_SIZE]
    chunk_df = spark.createDataFrame(chunk_pdf, schema=emb_schema)
    write_mode = "overwrite" if i == 0 else "append"
    (chunk_df.write
        .mode(write_mode)
        .option("overwriteSchema", "true" if i == 0 else "false")
        .saveAsTable(EMB_TABLE))
    print(f"  Written chunk {i // CHUNK_SIZE + 1}: rows {i}–{min(i + CHUNK_SIZE, len(src_pd)) - 1}")

spark.sql(f"""
    ALTER TABLE {EMB_TABLE}
    SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

try:
    spark.sql(f"ALTER TABLE {EMB_TABLE} ALTER COLUMN pr_id SET NOT NULL")
    spark.sql(f"ALTER TABLE {EMB_TABLE} ADD CONSTRAINT pk_pr_id PRIMARY KEY (pr_id)")
    print("PRIMARY KEY constraint declared.")
except Exception as e:
    if "already exists" in str(e).lower():
        print("PRIMARY KEY constraint already exists — skipping.")
    else:
        raise

row_count = spark.table(EMB_TABLE).count()
print(f"\n{EMB_TABLE}")
print(f"  Rows: {row_count:,}  |  dim: {EMB_DIM}  |  CDF: on  |  PK: pr_id")
print(f"  MAX_LEN: {MAX_LEN}  |  BATCH: {BATCH}  |  length-sorted batching: on")
print(f"  Vectors embed contextual_retrieval_text (fine | mid | core)")


In [0]:
# Cell 18 — Create AI Search endpoint + delta sync indexes
# ============================================================================
# SECTION J — AI SEARCH ENDPOINT + THREE DELTA SYNC INDEXES   [AISearchClient]
#
# Hybrid backbone. Uses your PRECOMPUTED BGE-M3 vectors
# (embedding_vector_column + embedding_dimension) — Databricks does NOT
# recompute embeddings.
#
# One index per vector column (Delta Sync requires this). Each index also
# syncs its tier's contextual_retrieval_text so AI Search's native keyword
# (Okapi BM25) side runs over EXACTLY the enriched text — no separate BM25
# index needed. At query time query_type="HYBRID" fuses vector + BM25 with
# Reciprocal Rank Fusion (rrf_param=60) inside AI Search.
#
#   embedding      → fine index   ← contextual_retrieval_text_full
#   mid_embedding  → mid  index   ← contextual_retrieval_text_mid
#   core_embedding → core index   ← contextual_retrieval_text_core
#
# Prereqs: EMB_TABLE has CDF enabled + a PRIMARY KEY on pr_id.
# NOTE: only pr_id + the tier's contextual_retrieval_text are synced, so BM25
#       scores the enriched text alone (source/deterministic/llm text is
#       already contained inside contextual_retrieval_text — syncing the parts
#       separately would double-count their terms).
# ============================================================================
client = AISearchClient()          # auto-detects notebook credentials

# (Re)state names in case Python was restarted by the SDK install cell.
EMB_TABLE       = f"{CATALOG}.{ALYT}.deviation_embeddings"
EMB_INDEX_FINE  = f"{CATALOG}.{ALYT}.deviation_emb_fine_idx"
EMB_INDEX_MID   = f"{CATALOG}.{ALYT}.deviation_emb_mid_idx"
EMB_INDEX_CORE  = f"{CATALOG}.{ALYT}.deviation_emb_core_idx"
EMB_DIM         = 1024
VS_ENDPOINT     = "deviation-retrieval-vs"

# Per-index searchable text: {vector_col -> contextual_retrieval_text tier}.
# This is the column AI Search's BM25 keyword side scores for that index.
TIER_TEXT = {
    "embedding":      "contextual_retrieval_text_full",
    "mid_embedding":  "contextual_retrieval_text_mid",
    "core_embedding": "contextual_retrieval_text_core",
}


# ── helper: read a normalized state string out of index.describe() ─────────
def _describe_state(desc: dict) -> str:
    """describe() returns a plain dict; probe its status defensively."""
    st = (desc or {}).get("status", {}) or {}
    return str(
        st.get("detailed_state")
        or ("ONLINE_READY" if st.get("ready") is True else None)
        or st.get("message")
        or "UNKNOWN"
    )


# ── 1. Create (or reuse) the endpoint ──────────────────────────────────────
try:
    client.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print(f"Creating endpoint '{VS_ENDPOINT}' …")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{VS_ENDPOINT}' already exists — reusing.")
    else:
        raise

# ── 2. Create-or-reuse one Delta Sync index per vector column ──────────────
def _ensure_index(index_name: str, vector_col: str) -> None:
    text_col = TIER_TEXT[vector_col]
    # Sync the key + this tier's enriched text so HYBRID/BM25 scores that text.
    columns_to_sync = ["pr_id", text_col]
    try:
        kwargs = dict(
            endpoint_name=VS_ENDPOINT,
            index_name=index_name,
            source_table_name=EMB_TABLE,
            primary_key="pr_id",
            pipeline_type="TRIGGERED",
            embedding_dimension=EMB_DIM,          # existing-embeddings path
            embedding_vector_column=vector_col,   # our precomputed BGE-M3 column
        )
        try:
            client.create_delta_sync_index(columns_to_sync=columns_to_sync, **kwargs)
        except TypeError:
            # Older/newer SDK may not expose columns_to_sync → sync all columns.
            client.create_delta_sync_index(**kwargs)
        print(f"  Created : {index_name}  ({vector_col} + {text_col})  — self-syncs on init")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"  Exists  : {index_name}  ({vector_col} + {text_col})")
        else:
            raise

    # Poll describe() until ONLINE
    for attempt in range(60):
        try:
            desc = client.get_index(endpoint_name=VS_ENDPOINT,
                                    index_name=index_name).describe()
        except Exception as e:
            print(f"    [{attempt+1:02d}] describe() not ready: {e}")
            time.sleep(30); continue
        state = _describe_state(desc)
        print(f"    [{attempt+1:02d}] {state}")
        if "ONLINE" in state:
            return
        if "FAILED" in state:
            raise RuntimeError(f"Index '{index_name}' entered FAILED: {desc.get('status')}")
        time.sleep(30)
    raise TimeoutError(f"Index '{index_name}' did not reach ONLINE.")


print("\n── Fine index (embedding — contextual_retrieval_text_full) ──")
_ensure_index(EMB_INDEX_FINE, "embedding")

print("\n── Mid index (mid_embedding — contextual_retrieval_text_mid) ──")
_ensure_index(EMB_INDEX_MID, "mid_embedding")

print("\n── Core index (core_embedding — contextual_retrieval_text_core) ──")
_ensure_index(EMB_INDEX_CORE, "core_embedding")

print(f"\nAI Search stack ready (vector + native BM25 hybrid)")
print(f"  Endpoint : {VS_ENDPOINT}")
print(f"  Fine idx : {EMB_INDEX_FINE}")
print(f"  Mid  idx : {EMB_INDEX_MID}")
print(f"  Core idx : {EMB_INDEX_CORE}")

In [0]:
# Cell 19 — Hybrid retrieval + rerank
# ============================================================================
# SECTION K — HYBRID RETRIEVAL + RERANK
#   AI Search native HYBRID (vector + BM25, RRF-fused) -> Top-N -> rerank -> Top-K
#
# STEP 4 of the article pipeline, across 3 tiers (core / mid / full):
#   1. Retrieve  : one query_type="HYBRID" call per tier. AI Search runs
#                  vector similarity over the tier's embedding AND Okapi BM25
#                  over its contextual_retrieval_text, then fuses both with
#                  Reciprocal Rank Fusion (rrf_param=60) internally.
#   2. Over-retrieve Top-N : ask for RERANK_CANDIDATES fused hits (10x final K).
#   3. Rerank    : (disabled) — RRF recall is already relevance-ranked.
#   4. Return Top-K : highest reranked results.
#
# Because the index stores precomputed BGE-M3 vectors (self-managed
# embeddings), the HYBRID call supplies BOTH query_vector (vector side) and
# query_text (BM25 side). No hand-rolled BM25 index or RRF needed.
#
# NOTE: if Python was restarted by the SDK-install cell, `bge` is gone; the
#       reload block below brings it back (pinned to a single GPU to avoid the
#       FlagEmbedding multi-GPU DataParallel replicate stall).
# ============================================================================

# --- silence the benign torchao cpp-extension fallback warning (not our bug) ---
warnings.filterwarnings("ignore", message=".*cpp extensions.*")
warnings.filterwarnings("ignore", message=".*torchao.*")

client = AISearchClient()

# --- reload bge (dense encoder) on ONE gpu if it was wiped by restartPython() ---
if "bge" not in globals():
    from FlagEmbedding import BGEM3FlagModel
    print("Reloading BAAI/bge-m3 on cuda:0 …")
    try:
        bge = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, devices="cuda:0")   # pin (newer API)
    except TypeError:
        bge = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, device="cuda:0")    # pin (older API)
    print("Dense model reloaded (single GPU).")

# --- source-text lookup for printing result snippets ---
_meta = {r["pr_id"]: (r["source_free_text_full"] or "")
         for r in spark.table(EMB_TABLE).select("pr_id", "source_free_text_full").collect()}

MAX_LEN     = 8192
NUM_RESULTS = 10
QUERY       = "site inadvertently disclosed treatment assignment, possible unblinding"
TIER_INDEX  = {"full": EMB_INDEX_FINE, "mid": EMB_INDEX_MID, "core": EMB_INDEX_CORE}


class HybridRetriever:
    """Hybrid (vector + BM25, RRF-fused) retrieval over the tiered AI Search
    indexes, with an optional cross-encoder rerank stage.

    Encapsulates the AI Search client, the tier→index map, the BGE-M3 query
    encoder, and the recall/rerank pipeline behind `.search(query, tier)`.
    """

    def __init__(self, client, bge, tier_index, endpoint, *,
                 num_results=10, over_retrieve=10, max_len=8192):
        self.client = client
        self.bge = bge
        self.tier_index = tier_index
        self.endpoint = endpoint
        self.num_results = num_results
        self.rerank_candidates = num_results * over_retrieve   # over-retrieve for rerank
        self.max_len = max_len

    def _embed(self, text: str) -> list:
        out = self.bge.encode(
            [text], batch_size=1, max_length=self.max_len,
            return_dense=True, return_sparse=False, return_colbert_vecs=False,
        )
        return out["dense_vecs"][0].tolist()

    def _recall(self, tier: str, query: str, query_vec: list) -> list:
        """RRF-fused (vector + BM25) pr_ids from AI Search's native HYBRID search."""
        index = self.client.get_index(endpoint_name=self.endpoint,
                                      index_name=self.tier_index[tier])
        res = index.similarity_search(
            query_vector=query_vec,     # vector side (self-managed embeddings)
            query_text=query,           # BM25 keyword side
            query_type="HYBRID",        # AI Search fuses both with RRF (rrf_param=60)
            columns=["pr_id"],
            num_results=self.rerank_candidates,
        )
        rows = res.get("result", {}).get("data_array", []) if isinstance(res, dict) else []
        return [str(row[0]) for row in rows]

    def _rerank(self, candidate_ids: list) -> list:
        """Stage-2 rerank DISABLED — positional scores (recall is already RRF-sorted)."""
        n = len(candidate_ids)
        return [(pid, 1.0 - i / n) for i, pid in enumerate(candidate_ids)]

    def search(self, query: str, tier: str) -> list:
        query_vec  = self._embed(query)
        candidates = self._recall(tier, query, query_vec)
        print(f"    [{tier}] hybrid recall returned {len(candidates)} candidate(s)")  # catch silent empty recall
        return self._rerank(candidates)[:self.num_results]


retriever = HybridRetriever(client, bge, TIER_INDEX, VS_ENDPOINT,
                            num_results=NUM_RESULTS, over_retrieve=10, max_len=MAX_LEN)

for tier in ["core", "mid", "full"]:
    print(f"── HYBRID+Rerank({tier})  query: '{QUERY}' ──")
    results = retriever.search(QUERY, tier)
    if not results:
        print("  (no results — recall was empty; check index name / VS_ENDPOINT for this tier)")
    for pid, score in results:
        snippet = (_meta.get(pid) or "").replace("\n", " ")[:120]
        print(f"  {score:.4f}  [pr_id={pid}]  {snippet}")
    print()

In [0]:
# Cell 20 — Diagnostics: embedding table + index status
# ============================================================================
# DIAGNOSTICS   [AISearchClient]  — embedding table + AI Search index status
# ============================================================================

client = AISearchClient()

def _describe_state(desc: dict) -> str:
    st = (desc or {}).get("status", {}) or {}
    return str(
        st.get("detailed_state")
        or ("ONLINE_READY" if st.get("ready") is True else None)
        or st.get("message") or "UNKNOWN"
    )

def _describe_rows(desc: dict):
    st = (desc or {}).get("status", {}) or {}
    return st.get("indexed_row_count", "n/a")

# ── 1. Embedding table: rows, PK integrity, dims, enrichment coverage ──────
try:
    df = spark.table(EMB_TABLE)
    row_count = df.count()
    sample = df.select(
        F.size("embedding").alias("fine_dim"),
        F.size("mid_embedding").alias("mid_dim"),
        F.size("core_embedding").alias("core_dim"),
    ).first()
    cov = df.select(
        F.count("*").alias("total"),
        F.sum(F.when(F.length(F.trim("deterministic_context")) > 0, 1).otherwise(0)).alias("with_ctx"),
        F.countDistinct("pr_id").alias("distinct_pr_id"),
    ).first()
    pct = (100.0 * cov["with_ctx"] / cov["total"]) if cov["total"] else 0.0

    print(f"{EMB_TABLE}")
    print(f"  Rows            : {row_count:,}")
    print(f"  Distinct pr_id  : {cov['distinct_pr_id']:,}  (should == rows)")
    print(f"  dims (f/m/c)    : {sample['fine_dim']} / {sample['mid_dim']} / {sample['core_dim']}")
    print(f"  With context    : {cov['with_ctx']:,} / {cov['total']:,}  ({pct:.1f}%)")
    if cov["distinct_pr_id"] != row_count:
        print("  WARNING: Duplicate pr_id — PRIMARY KEY assumption violated.")
    if EMB_DIM not in (sample["fine_dim"], sample["mid_dim"], sample["core_dim"]):
        print(f"  WARNING: A tier's dim != {EMB_DIM}.")
    if pct == 0.0:
        print("  WARNING: No deterministic context — check the source-resolution cell.")
except Exception as e:
    print(f"ERROR: Embedding table missing or empty: {e}")

# ── 2. AI Search index status ──────────────────────────────────────────────
print()
for label, idx_name in [("FINE", EMB_INDEX_FINE), ("MID", EMB_INDEX_MID), ("CORE", EMB_INDEX_CORE)]:
    try:
        desc = client.get_index(endpoint_name=VS_ENDPOINT, index_name=idx_name).describe()
        print(f"  [{label:4}]  {_describe_state(desc)}  |  indexed rows: {_describe_rows(desc)}")
    except Exception as e:
        print(f"  [{label:4}]  NOT FOUND — {e}")
